# MobileNet V2 Model

In [1]:
# 1. Install Kaggle CLI
!pip install -q kaggle

In [2]:
!kaggle datasets download -d masoudnickparvar/brain-tumor-mri-dataset

Dataset URL: https://www.kaggle.com/datasets/masoudnickparvar/brain-tumor-mri-dataset
License(s): Attribution 4.0 International (CC BY 4.0)
brain-tumor-mri-dataset.zip: Skipping, found more recently modified local copy (use --force to force download)


In [3]:
# Unzip the images
!unzip -qn brain-tumor-mri-dataset.zip -d brain_tumor_data

In [4]:
# Check the directory structure
print("Training data:")
!cd brain_tumor_data && cd Training && ls
print("\nTesting data:")
!cd brain_tumor_data && cd Testing && ls

Training data:
glioma	meningioma  notumor  pituitary

Testing data:
glioma	meningioma  notumor  pituitary


In [5]:
import getpass

github_token = getpass.getpass("Enter github token to clone the repository: ")

# Cloning the repo
!git clone https://{github_token}@github.com/Dini-s/brain-tumour-mri-classification.git

fatal: destination path 'brain-tumour-mri-classification' already exists and is not an empty directory.


In [6]:
# add the path to the cloned repo to sys.path
import sys
sys.path.append('brain-tumour-mri-classification')

#importing the modules within src folder
from src.dataset import get_dataloaders
from src.utils import confusion_matrix, print_metrics, plot_confusion_matrix, plot_learning_curves, evaluate_model
from src.transforms import train_transforms, val_test_transforms

#importing the required libraries
import torch
import torch.nn as nn
import torch.optim as optimizations
import torchvision.models as models
import time
from pathlib import Path

# <h3> Settings for the whole notebook <h3>

In [7]:
DATA_DIR = '/content/brain_tumor_data'
BATCH_SIZE = 32
SEED = 42
NUM_CLASSES = 4
LR_PHASE1 = 0.001
LR_PHASE2 = 0.0001
EPOCHS_PHASE1 = 10
EPOCHS_PHASE2 = 15
CHECKPOINT_DIR = Path('/content/mobilenetv2_best.pth')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# <h3> Loading Data <h3>

In [8]:
training, validation, test = get_dataloaders(DATA_DIR, BATCH_SIZE)

image, label = next(iter(training))

print(f"Image shape: {image.shape}")
print(f"Label: {label}")

print(f"Number of images in the batch: {len(image)}")

Train:      4,480 images
Validation: 1,120 images
Test:       1,600 images
Classes:    ['glioma', 'meningioma', 'notumor', 'pituitary']
Image shape: torch.Size([32, 3, 224, 224])
Label: tensor([0, 0, 0, 0, 2, 3, 0, 2, 0, 1, 0, 0, 0, 2, 2, 3, 0, 3, 1, 0, 3, 3, 0, 1,
        1, 2, 1, 0, 3, 2, 0, 1])
Number of images in the batch: 32


# <h3>Building the Model</h3>

importing the model and replacing the classifier layer with a new layer. The new layer has 4 output classes

In [9]:
from torchvision.models import mobilenet_v2, MobileNet_V2_Weights

model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT, progress=True)

classifier = model.classifier
input_features = classifier[1].in_features

# setting the manula seed
torch.manual_seed(SEED)

# new layer with 4 output classes
classifier[1] = nn.Linear(in_features=input_features, out_features=NUM_CLASSES, bias=True)

print(classifier)

Sequential(
  (0): Dropout(p=0.2, inplace=False)
  (1): Linear(in_features=1280, out_features=4, bias=True)
)


counting the number of trainable parameters in the model

In [10]:
global COUNT

def count_parameters(model):
    COUNT = 0
    for parameters in model.parameters():
        if parameters.requires_grad:
            COUNT += parameters.numel()

    print(f"Total number of trainable parameters: {COUNT}")

count_parameters(model)

Total number of trainable parameters: 2228996


<b>Running the model on a batch of images in evaluation mode and without gradients</b>

In [11]:
# Moving the model and data to device
moved_model = model.to(DEVICE)
moved_images_batch = image.to(DEVICE)
moved_labels = label.to(DEVICE)
# 1 test forward pass
with torch.no_grad():
    moved_model.eval()
    output = moved_model(moved_images_batch)

<b>Calculating the loss</b>

In [12]:
loss_function = nn.CrossEntropyLoss()
loss = loss_function(output, moved_labels)
print(f"Loss: {loss.item()}")

Loss: 1.3681714534759521


# <b>Training</b>

<b>Training will be done in 2 phases. In first phase, we will freeze the feature extractor and only train the classifier.</b>

<b>Freezing the feature extractor</b>

In [13]:
for parameters in moved_model.features.parameters():
    parameters.requires_grad = False # Freezing the feature extractor

count_parameters(moved_model)

Total number of trainable parameters: 5124


In [14]:
criterion = nn.CrossEntropyLoss()
optimizer = optimizations.Adam(moved_model.classifier.parameters(), LR_PHASE1)

<b>Function to train the model for one epoch</b>

In [15]:
def train_one_epoch(model, loader, criterion, optimizer, device):

    model.to(device)
    model.train()

    total_training_loss = 0.0
    number_of_images = 0
    total_correct_predictions = 0

    for images, labels in loader:

        print("Processing Batch...")

        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        # Forward pass
        output = model(images)

        loss = criterion(output, labels)
        loss.backward()
        optimizer.step()

        number_of_images += images.size(0)

        total_training_loss_per_batch = loss.item() * images.size(0)

        print(f"Total training loss for batch: {total_training_loss_per_batch}")

        total_training_loss += total_training_loss_per_batch

        predictions = output.argmax(dim=1)

        total_correct_for_batch = (labels == predictions).sum().item()

        total_correct_predictions += total_correct_for_batch

    print("Finished training epoch.")

    averge_training_loss = total_training_loss / number_of_images
    accuracy = (total_correct_predictions / number_of_images) * 100

    return averge_training_loss, accuracy

<b>Evaluate one epoch</b>

In [16]:
def evaluate_one_epoch(model, loader, criterion, device):

    model.to(device)
    with torch.no_grad():
        model.eval()

        total_training_loss = 0.0
        number_of_images = 0
        total_correct_predictions = 0

        for images, labels in loader:

            print("Processing Batch...")

            images, labels = images.to(device), labels.to(device)

            # Forward pass
            output = model(images)

            loss = criterion(output, labels)

            number_of_images += images.size(0)

            total_training_loss_per_batch = loss.item() * images.size(0)

            print(f"Total training loss for batch: {total_training_loss_per_batch}")

            total_training_loss += total_training_loss_per_batch

            predictions = output.argmax(dim=1)

            total_correct_for_batch = (labels == predictions).sum().item()

            total_correct_predictions += total_correct_for_batch

        print("Finished evaluation of epoch.")

        averge_training_loss = total_training_loss / number_of_images
        accuracy = (total_correct_predictions / number_of_images) * 100

        return averge_training_loss, accuracy

In [17]:
def train_model(model, training_loader, validation_loader, criterion, optimizer, device, epochs, checkpoint_path):

    train_losses = []
    validation_losses = []
    train_accuracies = []
    validation_accuracies = []

    best_validation_accuracy = 0.0

    start_time = time.time()

    for epoch in range(epochs):
        print(f"Epoch {epoch + 1}/{epochs}")

        train_loss, train_accuracy = train_one_epoch(model, training_loader, criterion, optimizer, device)
        val_loss, val_accuracy = evaluate_one_epoch(model, validation_loader, criterion, device)

        train_losses.append(train_loss)
        validation_losses.append(val_loss)
        train_accuracies.append(train_accuracy)
        validation_accuracies.append(val_accuracy)

        print(f"Training Loss: {train_loss:.4f}, Training Accuracy: {train_accuracy:.2f}%")
        print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_accuracy:.2f}%")

        if (val_accuracy > best_validation_accuracy):
            torch.save(model.state_dict(), checkpoint_path)
            best_validation_accuracy = val_accuracy

    end_time = time.time()
    phase1_seconds = end_time - start_time

    print(f"Phase 1 Time: {phase1_seconds:.2f} seconds")
    print(f"Best Validation Accuracy: {best_validation_accuracy:.2f}%")
    print(f"Number of epochs: {epochs}")

In [18]:
train_model(moved_model, training, validation, criterion, optimizer, DEVICE, EPOCHS_PHASE1, CHECKPOINT_DIR)

Epoch 1/10
Processing Batch...
Total training loss for batch: 44.92610168457031
Processing Batch...
Total training loss for batch: 44.21595764160156
Processing Batch...
Total training loss for batch: 43.84022521972656
Processing Batch...
Total training loss for batch: 41.749454498291016
Processing Batch...
Total training loss for batch: 42.24635314941406
Processing Batch...
Total training loss for batch: 39.60888671875
Processing Batch...
Total training loss for batch: 40.66501235961914
Processing Batch...
Total training loss for batch: 40.57438278198242
Processing Batch...
Total training loss for batch: 41.356327056884766
Processing Batch...
Total training loss for batch: 37.54343032836914
Processing Batch...
Total training loss for batch: 37.791351318359375
Processing Batch...
Total training loss for batch: 36.2286376953125
Processing Batch...
Total training loss for batch: 35.87625503540039
Processing Batch...
Total training loss for batch: 37.34139633178711
Processing Batch...
Tota

<b></b>